# TIL-AI 2026 NLP SmolLM3 Answer Tuning

This notebook fine-tunes `HuggingFaceTB/SmolLM3-3B` as the answer generator used after hybrid retrieval and reranking in `nlp_manager.py`.

The runtime container should use the merged output folder at `nlp/src/models/smollm3_answer_model`. Training uses single-document positive and hard-negative contexts so runtime can try the top 3 documents individually and skip unsupported candidates with `NO_ANSWER`. The notebook is Colab-first and follows the repo's `/content/drive/MyDrive/til26_data/nlp` data/artifact convention.

In [1]:
!pip install -q -U "transformers>=4.57.0" "datasets>=3.0.0" "accelerate>=1.0.0" "peft>=0.15.2" "bitsandbytes>=0.44.0" "safetensors>=0.5.0" "torchao>=0.16.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 127.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.8 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 109.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 51.1 MB/s eta 0:00:00:00:0100:01


In [2]:
from __future__ import annotations

import json
import math
import os
import random
import re
import shutil
from collections import Counter
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)


In [3]:
BASE_MODEL = "HuggingFaceTB/SmolLM3-3B"
OUTPUT_DIR = Path("/content/smollm3_answer_lora")

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Not running in Colab or Drive already mounted:', exc)

def find_repo_root():
    starts = [Path.cwd(), Path('/content'), Path('/content/til-26-overflow')]
    for start in starts:
        if not start.exists():
            continue
        for candidate in [start, *start.parents]:
            if (candidate / 'nlp' / 'src').exists() and ((candidate / 'CODEX.md').exists() or (candidate / 'data' / 'nlp').exists()):
                return candidate
    return Path('/content')

repo_root = find_repo_root()
shortcut_path = Path('/content/drive/MyDrive/til26_data')
drive_data_dir = shortcut_path / 'nlp'
repo_data_dir = repo_root / 'data' / 'nlp'
DATA_DIR = Path(os.getenv('NLP_DATA_DIR', drive_data_dir if (drive_data_dir / 'nlp.jsonl').exists() else repo_data_dir))
if not (DATA_DIR / 'nlp.jsonl').exists():
    raise FileNotFoundError(f'Could not find nlp.jsonl under {DATA_DIR}. Mount Drive or set NLP_DATA_DIR.')

artifact_dir = shortcut_path / 'nlp' / 'models' / 'smollm3_answer_model'
repo_artifact_dir = repo_root / 'nlp' / 'src' / 'models' / 'smollm3_answer_model'

MAX_CHUNK_WORDS = 170
CHUNK_OVERLAP_WORDS = 35
RETRIEVAL_TOP_K = 24
TRAIN_DOCS_PER_QUESTION = int(os.getenv('TRAIN_DOCS_PER_QUESTION', '3'))
CHUNKS_PER_DOC_CONTEXT = int(os.getenv('CHUNKS_PER_DOC_CONTEXT', '3'))
NEGATIVES_PER_QUESTION = int(os.getenv('NEGATIVES_PER_QUESTION', '2'))
NO_ANSWER_TOKEN = os.getenv('NO_ANSWER_TOKEN', 'NO_ANSWER')
MAX_CONTEXT_CHARS = 3600
MAX_LENGTH = 4096

TRAIN_EPOCHS = float(os.getenv('TRAIN_EPOCHS', '2'))
LEARNING_RATE = float(os.getenv('LEARNING_RATE', '2e-4'))
BATCH_SIZE = int(os.getenv('BATCH_SIZE', '1'))
GRAD_ACCUM = int(os.getenv('GRAD_ACCUM', '8'))
MERGE_ADAPTER = os.getenv('MERGE_ADAPTER', '1') != '0'

print('repo_root:', repo_root)
print('data_dir:', DATA_DIR)
print('artifact_dir:', artifact_dir)
print('repo_artifact_dir:', repo_artifact_dir)
print('BASE_MODEL:', BASE_MODEL)


Mounted at /content/drive
repo_root: /content
data_dir: /content/drive/MyDrive/til26_data/nlp
artifact_dir: /content/drive/MyDrive/til26_data/nlp/models/smollm3_answer_model
repo_artifact_dir: /content/nlp/src/models/smollm3_answer_model
BASE_MODEL: HuggingFaceTB/SmolLM3-3B


In [4]:
TOKEN_RE = re.compile(r"\d+(?:[.,]\d+)?%?|[a-z0-9]+(?:[-'][a-z0-9]+)*", re.I)
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "did", "do", "does", "for",
    "from", "had", "has", "have", "how", "in", "into", "is", "it", "of", "on", "or",
    "the", "their", "to", "was", "were", "what", "when", "where", "which", "who",
    "whom", "whose", "why", "with",
}
EXPANSION_GROUPS = {
    "amount": {"amount", "budget", "cost", "costs", "funding", "much", "paid", "payment", "price", "revenue", "spend", "spent", "value", "valued", "worth"},
    "count": {"count", "many", "number", "quantity", "total"},
    "date": {"date", "day", "deadline", "month", "schedule", "time", "timeline", "when", "year"},
    "person": {"accountable", "ceo", "chair", "chief", "commander", "director", "founder", "head", "leader", "led", "manager", "officer", "person", "who"},
    "place": {"city", "country", "district", "facility", "location", "place", "region", "site", "where", "zone"},
    "negation": {"absent", "missing", "not", "noted", "omitted", "stated", "undisclosed", "unspecified"},
}

def normalize_space(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

def tokens(text: str) -> list[str]:
    return [match.group(0).lower() for match in TOKEN_RE.finditer(text)]

def question_cue_groups(question: str) -> set[str]:
    lower = question.lower()
    terms = set(tokens(question))
    groups = set()
    if "how many" in lower:
        groups.add("count")
    if "how much" in lower:
        groups.add("amount")
    for phrase, group in (("when", "date"), ("where", "place"), ("who", "person"), ("whom", "person"), ("whose", "person")):
        if phrase in lower:
            groups.add(group)
    for group, group_terms in EXPANSION_GROUPS.items():
        if terms & group_terms:
            groups.add(group)
    if re.search(r"\b(no|not|never|without|missing|unstated|unmentioned)\b", lower):
        groups.add("negation")
    return groups

def expanded_query(question: str) -> str:
    parts = [question]
    for group in question_cue_groups(question):
        parts.extend(sorted(EXPANSION_GROUPS[group]))
    content_terms = [term for term in tokens(question) if term not in STOPWORDS]
    if content_terms:
        parts.append(" ".join(content_terms))
    return " ".join(parts)

In [5]:
def document_title(document: str, doc_id: int) -> str:
    for line in document.splitlines():
        line = normalize_space(line).strip("#* ")
        if line:
            return line[:180]
    return f"Document {doc_id + 1}"

def chunk_document(document: str, doc_id: int, doc_name: str) -> list[dict]:
    document = document.replace("\r\n", "\n").replace("\r", "\n")
    title = document_title(document, doc_id)
    paragraphs = [normalize_space(part) for part in re.split(r"\n\s*\n", document) if normalize_space(part)]
    chunks = []
    current_words = []
    chunk_id = 0

    def flush():
        nonlocal chunk_id, current_words
        if not current_words:
            return
        body = " ".join(current_words)
        text = f"{title}\n{body}" if title and title not in body[:160] else body
        chunks.append({"doc_id": doc_id, "doc_name": doc_name, "chunk_id": chunk_id, "title": title, "text": text})
        chunk_id += 1
        current_words = current_words[-CHUNK_OVERLAP_WORDS:] if CHUNK_OVERLAP_WORDS > 0 else []

    stride = max(1, MAX_CHUNK_WORDS - CHUNK_OVERLAP_WORDS)
    for paragraph in paragraphs:
        words = paragraph.split()
        if len(words) > MAX_CHUNK_WORDS:
            flush()
            for start in range(0, len(words), stride):
                window = words[start:start + MAX_CHUNK_WORDS]
                if window:
                    chunks.append({"doc_id": doc_id, "doc_name": doc_name, "chunk_id": chunk_id, "title": title, "text": f"{title}\n{' '.join(window)}"})
                    chunk_id += 1
            current_words = []
            continue
        if current_words and len(current_words) + len(words) > MAX_CHUNK_WORDS:
            flush()
        current_words.extend(words)
    flush()
    return chunks

documents = []
doc_names = []
for path in sorted((DATA_DIR / "documents").glob("*.txt")):
    doc_names.append(path.stem)
    documents.append(path.read_text(encoding="utf-8"))

chunks = []
for doc_id, (doc_name, document) in enumerate(zip(doc_names, documents)):
    chunks.extend(chunk_document(document, doc_id, doc_name))
for chunk_index, chunk in enumerate(chunks):
    chunk["index"] = chunk_index
chunks_by_doc_name = {}
for chunk in chunks:
    chunks_by_doc_name.setdefault(chunk["doc_name"], []).append(chunk)

doc_freq = Counter()
chunk_terms = []
chunk_lengths = []
for chunk in chunks:
    terms = Counter(tokens(chunk["text"]))
    chunk_terms.append(terms)
    chunk_lengths.append(sum(terms.values()))
    doc_freq.update(terms.keys())

avgdl = sum(chunk_lengths) / max(1, len(chunk_lengths))
idf = {term: math.log(1.0 + (len(chunks) - freq + 0.5) / (freq + 0.5)) for term, freq in doc_freq.items()}
print(f"Loaded {len(documents)} documents and {len(chunks)} chunks")

Loaded 296 documents and 2555 chunks


In [6]:
def bm25_query_terms(question: str) -> Counter:
    return Counter(tokens(expanded_query(question)))

def score_chunk_index(query_terms: Counter, index: int) -> float:
    terms = chunk_terms[index]
    score = 0.0
    k1 = 1.45
    b = 0.72
    length = chunk_lengths[index] or 1
    length_norm = k1 * (1.0 - b + b * length / avgdl)
    for term, query_count in query_terms.items():
        frequency = terms.get(term, 0)
        if not frequency:
            continue
        weight = 0.35 if term in STOPWORDS else 1.0
        tf = (frequency * (k1 + 1.0)) / (frequency + length_norm)
        score += idf.get(term, 0.0) * tf * weight * min(2.0, query_count)
    return score

def retrieve_bm25(question: str, k: int = RETRIEVAL_TOP_K) -> list[dict]:
    query_terms = bm25_query_terms(question)
    if not query_terms:
        return []
    scores = []
    for index, _terms in enumerate(chunk_terms):
        score = score_chunk_index(query_terms, index)
        if score > 0:
            scores.append((index, score))
    scores.sort(key=lambda item: item[1], reverse=True)
    return [chunks[index] for index, _score in scores[:k]]

def chunks_for_doc_name(doc_name: str, question: str, chunk_limit: int = CHUNKS_PER_DOC_CONTEXT) -> list[dict]:
    doc_chunks = chunks_by_doc_name.get(doc_name, [])
    if not doc_chunks:
        return []
    query_terms = bm25_query_terms(question)
    scored = []
    for chunk in doc_chunks:
        score = score_chunk_index(query_terms, chunk["index"])
        if score > 0:
            scored.append((score, chunk))
    scored.sort(key=lambda item: item[0], reverse=True)
    selected = [chunk for _score, chunk in scored[:chunk_limit]]
    return selected or doc_chunks[:chunk_limit]

def top_document_groups(question: str, doc_limit: int = TRAIN_DOCS_PER_QUESTION, chunk_limit: int = CHUNKS_PER_DOC_CONTEXT) -> list[list[dict]]:
    retrieved = retrieve_bm25(question, k=max(RETRIEVAL_TOP_K, doc_limit * 8))
    selected_doc_names = []
    seen = set()
    for chunk in retrieved:
        doc_name = chunk["doc_name"]
        if doc_name in seen:
            continue
        seen.add(doc_name)
        selected_doc_names.append(doc_name)
        if len(selected_doc_names) >= doc_limit:
            break
    groups = []
    for doc_name in selected_doc_names:
        group = [chunk for chunk in retrieved if chunk["doc_name"] == doc_name][:chunk_limit]
        if group:
            groups.append(group)
    return groups

def build_context_from_chunks(doc_chunks: list[dict]) -> str:
    blocks = []
    used_chars = 0
    for evidence_index, chunk in enumerate(doc_chunks, start=1):
        block = normalize_space(chunk["text"])
        block = f"[{evidence_index}] {chunk['title']}: {block}"
        if used_chars + len(block) > MAX_CONTEXT_CHARS:
            remaining = MAX_CONTEXT_CHARS - used_chars
            if remaining <= 240:
                break
            block = block[:remaining].rsplit(" ", 1)[0]
        blocks.append(block)
        used_chars += len(block)
    return "\n".join(blocks)

def build_context(question: str) -> str:
    groups = top_document_groups(question, doc_limit=1)
    return build_context_from_chunks(groups[0]) if groups else ""

In [7]:
instances = []
with (DATA_DIR / "nlp.jsonl").open("r", encoding="utf-8") as handle:
    for line in handle:
        if line.strip():
            instances.append(json.loads(line))

SYSTEM_PROMPT = (
    "/no_think\n"
    "You answer retrieval questions using only evidence from one candidate document. "
    "Return one concise answer string. Do not explain. "
    "For arithmetic or comparisons, compute the answer from the evidence. "
    f"If the evidence does not answer the question, return {NO_ANSWER_TOKEN}. "
    "Do not mention the evidence or context."
)

def make_user_prompt(question: str, context: str) -> str:
    return (
        f"Question: {question}\n\n"
        f"Evidence:\n{context}\n\n"
        "Answer with the shortest phrase or sentence that fully answers the question. "
        f"If the evidence does not answer it, return {NO_ANSWER_TOKEN}:"
    )

def source_doc_names(instance: dict) -> list[str]:
    return [str(doc).strip() for doc in (instance.get("source_docs") or []) if str(doc).strip()]

def make_training_rows(instance: dict) -> list[dict]:
    question = instance["question"].strip()
    raw_answer = instance.get("answer")
    answer = "" if raw_answer is None else str(raw_answer).strip()
    if not question or not answer:
        return []

    source_docs = set(source_doc_names(instance))
    rows_for_question = []
    seen = set()

    def add_row(doc_name: str, doc_chunks: list[dict], target: str) -> None:
        if not doc_chunks:
            return
        context = build_context_from_chunks(doc_chunks)
        if not context:
            return
        key = (doc_name, target)
        if key in seen:
            return
        seen.add(key)
        rows_for_question.append({
            "question": question,
            "doc_name": doc_name,
            "context": context,
            "answer": target,
            "user_prompt": make_user_prompt(question, context),
        })

    for doc_name in sorted(source_docs):
        add_row(doc_name, chunks_for_doc_name(doc_name, question), answer)

    negative_count = 0
    for group in top_document_groups(question):
        doc_name = group[0]["doc_name"]
        if source_docs and doc_name not in source_docs:
            if negative_count < NEGATIVES_PER_QUESTION:
                add_row(doc_name, group, NO_ANSWER_TOKEN)
                negative_count += 1
            continue
        add_row(doc_name, group, answer)

    if not rows_for_question:
        context = build_context(question)
        if context:
            rows_for_question.append({"question": question, "doc_name": "", "context": context, "answer": answer, "user_prompt": make_user_prompt(question, context)})
    return rows_for_question

rows = []
for instance in instances:
    rows.extend(make_training_rows(instance))
random.shuffle(rows)
split = max(1, int(0.9 * len(rows)))
train_rows = rows[:split]
eval_rows = rows[split:]
positive_rows = sum(1 for row in rows if row["answer"] != NO_ANSWER_TOKEN)
negative_rows = len(rows) - positive_rows
print(f"Rows: {len(rows)} | positives: {positive_rows} | hard negatives: {negative_rows}")
print(f"Train rows: {len(train_rows)} | Eval rows: {len(eval_rows)}")
print(train_rows[0]["user_prompt"][:1000])
print("ANSWER:", train_rows[0]["answer"])

Rows: 2649 | positives: 883 | hard negatives: 1766
Train rows: 2384 | Eval rows: 265
Question: Where did CGC geospatial analysis place the partially legible coordinates found in the SIGINT-76-0441 transmission?

Evidence:
[1] ```: ``` in three prior intercepted Conclave communications. The reference here suggests the described logistical activity is directed toward satisfying CYPHER's operational requirements. Analysts should cross-reference this annotation with Section A above when assessing mission purpose. --- **SECTION D — ANALYST ANNOTATION: COORDINATE FRAGMENT / GEOSPATIAL ANALYSIS** The partially legible coordinate string recovered from the final decrypted segment was submitted to CGC geospatial analysis. Resolution is insufficient for precise location identification; however, the recovered partial values place the referenced coordinates **within approximately 200 kilometers of Sarento**. Significance is unconfirmed. Analysts note that Sarento's position as a coastal industrial 

In [8]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def chat_prompt(user_prompt: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def tokenize_row(row: dict) -> dict:
    prompt = chat_prompt(row["user_prompt"])
    answer = row["answer"] + (tokenizer.eos_token or "")
    full_text = prompt + answer
    encoded = tokenizer(full_text, max_length=MAX_LENGTH, truncation=True)
    prompt_ids = tokenizer(prompt, max_length=MAX_LENGTH, truncation=True)["input_ids"]
    labels = encoded["input_ids"].copy()
    prompt_len = min(len(prompt_ids), len(labels))
    labels[:prompt_len] = [-100] * prompt_len
    encoded["labels"] = labels
    return encoded

train_dataset = Dataset.from_list(train_rows).map(tokenize_row, remove_columns=list(train_rows[0].keys()))
eval_dataset = Dataset.from_list(eval_rows).map(tokenize_row, remove_columns=list(eval_rows[0].keys())) if eval_rows else None
print(train_dataset[0].keys())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/2384 [00:00<?, ? examples/s]

Map:   0%|          | 0/265 [00:00<?, ? examples/s]

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [9]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quantization_config,
    device_map="auto",
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

trainable params: 30,228,480 || all params: 3,105,327,104 || trainable%: 0.9734


In [10]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=TRAIN_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch" if eval_dataset is not None and len(eval_dataset) else "no",
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    report_to="none",
    remove_unused_columns=False,
)

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collator,
)
trainer.train()
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss
1,0.647456,0.364412
2,0.257262,0.363926


('/content/smollm3_answer_lora/tokenizer_config.json',
 '/content/smollm3_answer_lora/chat_template.jinja',
 '/content/smollm3_answer_lora/tokenizer.json')

In [11]:
def get_preview_model():
    preview_model = globals().get("model") or globals().get("preview_model") or globals().get("merged_model")
    if preview_model is not None:
        preview_model.eval()
        return preview_model

    adapter_dir = Path(OUTPUT_DIR)
    if not (adapter_dir / "adapter_config.json").exists():
        raise FileNotFoundError(f"No live model variable and no LoRA adapter found at {adapter_dir}")

    import importlib.metadata
    import subprocess
    import sys
    try:
        torchao_version = importlib.metadata.version("torchao")
        major, minor, *_ = [int(part) for part in torchao_version.split(".")[:2]]
        if (major, minor) < (0, 16):
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "torchao>=0.16.0"])
            for module_name in list(sys.modules):
                if module_name == "torchao" or module_name.startswith("torchao."):
                    sys.modules.pop(module_name, None)
            print("Updated torchao for PEFT adapter loading.")
    except importlib.metadata.PackageNotFoundError:
        pass

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=quantization_config,
        device_map="auto",
    )
    preview_model = PeftModel.from_pretrained(base_model, str(adapter_dir))
    preview_model.eval()
    globals()["preview_model"] = preview_model
    return preview_model


def generate_answer(question: str, context: str | None = None) -> str:
    if context is None:
        context = build_context(question)
    user_prompt = make_user_prompt(question, context)
    prompt = chat_prompt(user_prompt)
    preview_model = get_preview_model()
    device = next(preview_model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = preview_model.generate(
            **inputs,
            max_new_tokens=96,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    answer_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(answer_ids, skip_special_tokens=True).strip()

for row in eval_rows[:5] if eval_rows else train_rows[:5]:
    print("Q:", row["question"])
    print("GT:", row["answer"])
    print("PRED:", generate_answer(row["question"], row["context"]))
    print("---")

Q: Given the projected annual revenue and the total development cost for Project Liminal, how many years of post-launch recurring revenue at the low end of projections would be needed to recoup the initial investment?
GT: NO_ANSWER


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


PRED: NO_ANSWER
---
Q: Given the incidents that prompted CGC Public Health Directive 119-07 and the definition of a mass disorientation event, what is the minimum total number of patrons who must have been simultaneously disoriented across all triggering incidents combined?
GT: NO_ANSWER
PRED: NO_ANSWER
---
Q: What share of the 347 intercepted non-Haven vessels during Q4 76 PCE were flagged for enhanced inspection, and how many ultimately resulted in seizure?
GT: NO_ANSWER
PRED: NO_ANSWER
---
Q: Why is the CGC Infrastructure Committee effectively unable to independently verify the resilience of the Edge network, despite conducting annual audits?
GT: Because the audits depend entirely on TEC self-reporting, and TEC's trade secret protections under the Cordial Entente block any independent access to the underlying infrastructure data.
PRED: TEC's refusal to share detailed infrastructure specifications with the CGC, combined with the committee's lack of technical staff and access to TEC's

In [12]:
if MERGE_ADAPTER:
    if "model" in globals():
        del model
    torch.cuda.empty_cache()
    import importlib.metadata
    import subprocess
    import sys
    try:
        torchao_version = importlib.metadata.version("torchao")
        major, minor, *_ = [int(part) for part in torchao_version.split(".")[:2]]
        if (major, minor) < (0, 16):
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "torchao>=0.16.0"])
            for module_name in list(sys.modules):
                if module_name == "torchao" or module_name.startswith("torchao."):
                    sys.modules.pop(module_name, None)
            print("Updated torchao for PEFT adapter merge.")
    except importlib.metadata.PackageNotFoundError:
        pass
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16,
        device_map="auto",
    )
    merged_model = PeftModel.from_pretrained(base_model, str(OUTPUT_DIR))
    merged_model = merged_model.merge_and_unload()
    artifact_dir.parent.mkdir(parents=True, exist_ok=True)
    if artifact_dir.exists():
        shutil.rmtree(artifact_dir)
    merged_model.config.use_cache = True
    merged_model.save_pretrained(str(artifact_dir), safe_serialization=True)
    tokenizer.save_pretrained(str(artifact_dir))

    if (repo_root / 'nlp' / 'src').exists():
        repo_artifact_dir.parent.mkdir(parents=True, exist_ok=True)
        if repo_artifact_dir.exists():
            shutil.rmtree(repo_artifact_dir)
        shutil.copytree(artifact_dir, repo_artifact_dir)

    zip_base = artifact_dir.parent / 'smollm3_answer_model'
    shutil.make_archive(str(zip_base), 'zip', artifact_dir)

    print('Saved model folder:', artifact_dir)
    print('Saved repo copy:', repo_artifact_dir)
    print('Saved zip:', str(zip_base) + '.zip')
else:
    artifact_dir.parent.mkdir(parents=True, exist_ok=True)
    zip_base = artifact_dir.parent / 'smollm3_answer_lora'
    shutil.make_archive(str(zip_base), 'zip', OUTPUT_DIR)
    print('Adapter saved to:', OUTPUT_DIR)
    print('Adapter zip:', str(zip_base) + '.zip')


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model folder: /content/drive/MyDrive/til26_data/nlp/models/smollm3_answer_model
Saved repo copy: /content/nlp/src/models/smollm3_answer_model
Saved zip: /content/drive/MyDrive/til26_data/nlp/models/smollm3_answer_model.zip


In [13]:
!ls drive/MyDrive/til26_data/nlp/models

nlp_answer_model      qwen_answer_model      smollm3_answer_model.zip
nlp_answer_model.zip  qwen_answer_model.zip
nlp_eval.zip	      smollm3_answer_model


In [14]:
!ls -lh /content/drive/MyDrive/til26_data/nlp/models/smollm3_answer_model.zip


-rw------- 1 root root 4.6G May 15 14:20 /content/drive/MyDrive/til26_data/nlp/models/smollm3_answer_model.zip


## Build Notes

1. The merged model is saved to Drive at `/content/drive/MyDrive/til26_data/nlp/models/smollm3_answer_model`.
2. If the repo is mounted in Colab, the notebook also copies it to `nlp/src/models/smollm3_answer_model`.
3. Before `til build nlp`, make sure that same folder exists in the Docker build context. `nlp_manager.py` will prefer it automatically, otherwise it falls back to base `HuggingFaceTB/SmolLM3-3B`.
4. Runtime loading uses `NLP_GENERATOR_QUANTIZATION=auto` by default, which attempts 4-bit bitsandbytes loading on CUDA and falls back to normal loading if quantization is unavailable.
